In [1]:
import pandas as pd

df = pd.read_parquet("../data/processed/fannie_2017_loan_level.parquet")

print(df.shape)
print(df["default_flag"].mean())               # your base rate — memorize this number
print(df.groupby("orig_quarter")["default_flag"].agg(["mean", "size"]))
print(df["LOAN_ID"].is_unique)                  # confirm the reduction did what it should

(2046851, 115)
0.034143178961243394
                  mean    size
orig_quarter                  
2017Q1        0.031214  487789
2017Q2        0.032269  492517
2017Q3        0.034398  546663
2017Q4        0.038399  519882
True


**Base rate: 3.4%** default across ~2.05M loans (2017 originations, all four quarters, stable 3.1%–3.8% by quarter). This is the reference point where all "lift" values below are subgroup rate ÷ base rate.

In [2]:
#check nulls and range
num_cols = ["CSCORE_B", "DTI", "ORIG_RATE", "OLTV", "OCLTV", "ORIG_UPB", "ORIG_TERM"]
for c in num_cols:
    coerced = pd.to_numeric(df[c], errors="coerce")
    print(f"{c:12s} nulls after coerce: {coerced.isna().mean():.3%}  range: {coerced.min()}–{coerced.max()}")
    df[c] = coerced

CSCORE_B     nulls after coerce: 0.077%  range: 445.0–850.0
DTI          nulls after coerce: 0.017%  range: 1.0–63.0
ORIG_RATE    nulls after coerce: 0.000%  range: 1.79–6.125
OLTV         nulls after coerce: 0.000%  range: 2–97
OCLTV        nulls after coerce: 0.000%  range: 2–114
ORIG_UPB     nulls after coerce: 0.000%  range: 5000.0–1223000.0
ORIG_TERM    nulls after coerce: 0.000%  range: 36–360


Data quality: all origination features coerce cleanly to numeric (<0.1% nulls); no Fannie sentinel values (9999 FICO, 999 DTI) contaminating the columns. Ranges are all plausible. Safe to model on.


In [3]:
def rate_by_bin(df, col, bins):
    g = df.groupby(pd.cut(df[col], bins))["default_flag"]
    out = g.agg(["mean", "size"])
    out["lift"] = out["mean"] / df["default_flag"].mean()
    return out

rate_by_bin(df, "CSCORE_B", [300, 620, 660, 700, 740, 780, 850])

/var/folders/5g/nyyrx9tx5nbfgnrym8rw723m0000gn/T/ipykernel_76784/3838274286.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = df.groupby(pd.cut(df[col], bins))["default_flag"]


,mean,size,lift
CSCORE_B,,,
"(300, 620]",0.123989,1484,3.631449
"(620, 660]",0.108010,102222,3.163443
"(660, 700]",0.073385,254412,2.149328
"(700, 740]",0.043869,422215,1.284844
"(740, 780]",0.023800,579244,0.697064
"(780, 850]",0.011140,685701,0.326285


In [4]:
df.groupby("PURPOSE")["default_flag"].agg(["mean", "size"]).sort_values("mean")
rate_by_bin(df, "DTI", [0, 20, 30, 36, 43, 50, 65])

/var/folders/5g/nyyrx9tx5nbfgnrym8rw723m0000gn/T/ipykernel_76784/3838274286.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = df.groupby(pd.cut(df[col], bins))["default_flag"]


,mean,size,lift
DTI,,,
"(0, 20]",0.011856,167425,0.347245
"(20, 30]",0.019133,484301,0.560368
"(30, 36]",0.029545,418380,0.865324
"(36, 43]",0.042624,601751,1.248388
"(43, 50]",0.054968,374633,1.609940
"(50, 65]",0.000000,21,0.000000


In [5]:
rate_by_bin(df, "OLTV", [0, 60, 70, 80, 90, 95, 100])

/var/folders/5g/nyyrx9tx5nbfgnrym8rw723m0000gn/T/ipykernel_76784/3838274286.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = df.groupby(pd.cut(df[col], bins))["default_flag"]


,mean,size,lift
OLTV,,,
"(0, 60]",0.020946,391141,0.613487
"(60, 70]",0.030740,252021,0.900312
"(70, 80]",0.031312,786434,0.917086
"(80, 90]",0.035643,233451,1.043941
"(90, 95]",0.051220,272568,1.500160
"(95, 100]",0.063280,111236,1.853368


In [6]:
df.groupby("STATE")["default_flag"].agg(["mean", "size"]).sort_values("mean", ascending=False).head(15)

,mean,size
STATE,,
VI,0.125000,128
PR,0.081258,2449
FL,0.056946,141082
NY,0.056179,62924
HI,0.053410,6759
LA,0.050447,20041
DC,0.048153,4901
TX,0.046143,164662
NJ,0.045474,47698


Risk hierarchy so far, strongest to weakest:

| Feature   | Pattern                          | Approx. swing | Notes |
|-----------|----------------------------------|---------------|-------|
| CSCORE_B  | Smooth, monotonic decline        | ~10×          | Dominant signal. <620 defaults 12.4% (3.6× lift); 780+ only 1.1% (0.33×). But <660 is a thin slice of the book — most loans are 740+. |
| DTI       | Smooth, monotonic rise           | ~4–5×         | No cliff at the 43 conforming limit; risk rises steadily with leverage. Independent of FICO, so additive. |
| OLTV      | Flat through 80, sharp tail rise | ~3×           | Signal is in the tail: ~3% up to 80% LTV, jumps to 5.1% (90–95) and 6.3% (95–100). Captures equity/skin-in-the-game. |
| PURPOSE   | Weak ordering                    | ~1.5×         | Cash-out refi (C) 3.8% > purchase (P) 3.5% > rate-term refi (R) 2.6%. Real but minor next to the above. |

Caveats for the team:
- **ORIG_RATE deliberately not treated as a predictor.** Rate is priced from the same risk assessed at origination, so using it to predict default is partly circular / leakage-adjacent. Documented as a relationship, not a feature. Revisit before modeling.
- With ~2M rows, every subgroup difference is "statistically significant" where we rely on **lift / effect size**, not p-values, to judge what matters.
- Always check the `size` column before trusting a rate (see the DTI 50+ bin: 0% default on only 21 loans = noise, not a finding).

## Features Added Parquet

The goal is to find which columns in our features added parquet are non-null and to find what will be useful for our ML models

In [7]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/processed/fannie_2017_features_added.parquet")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Base default rate: {df['default_flag'].mean():.4f}")

Rows: 2,046,851
Columns: 123
Base default rate: 0.0341


In [8]:
# Census: null count and share for every column, sorted worst-first.
# Fannie's public file carries many performance/recovery fields that are
# null for our origination-time book; this table documents exactly which.
census = (
    pd.DataFrame({
        "null_count": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(dropna=True),
    })
    .sort_values("null_pct", ascending=False)
)

pd.set_option("display.max_rows", 120)

# Headline buckets so the 115-column table is interpretable at a glance
fully_null   = census[census["null_pct"] == 100]
mostly_null  = census[(census["null_pct"] >= 50) & (census["null_pct"] < 100)]
partial_null = census[(census["null_pct"] > 0)  & (census["null_pct"] < 50)]
complete     = census[census["null_pct"] == 0]

print(f"Fully null (100%):        {len(fully_null)} columns")
print(f"Mostly null (50–99%):     {len(mostly_null)} columns")
print(f"Partially null (0–50%):   {len(partial_null)} columns")
print(f"Complete (0% null):       {len(complete)} columns")

census

Fully null (100%):        68 columns
Mostly null (50–99%):     3 columns
Partially null (0–50%):   3 columns
Complete (0% null):       49 columns


,null_count,null_pct,dtype,n_unique
PRINCIPAL_FORGIVENESS_AMOUNT,2046851,100.0,object,0
MISCELLANEOUS_HOLDING_EXPENSES_AND_CREDITS,2046851,100.0,object,0
ORIGINAL_LIST_START_DATE,2046851,100.0,object,0
POOL_ID,2046851,100.0,object,0
NON_INTEREST_BEARING_UPB,2046851,100.0,object,0
...,...,...,...,...
OCC_STAT,0,0.0,object,3
NO_UNITS,0,0.0,object,4
PROP,0,0.0,object,5
PURPOSE,0,0.0,object,3


In [9]:
# Sanity-check the engineered columns specifically — they should all be complete
engineered = ["interest_income_7yr", "lgd", "loss_if_default", "credit_grade",
              "is_first_time", "is_homeready", "is_hfa", "default_flag"]
census.loc[[c for c in engineered if c in census.index]]

,null_count,null_pct,dtype,n_unique
interest_income_7yr,0,0.0,float64,44122
lgd,0,0.0,float64,1
loss_if_default,0,0.0,float64,969
credit_grade,0,0.0,object,5
is_first_time,0,0.0,int8,2
is_homeready,0,0.0,int8,2
is_hfa,0,0.0,int8,2
default_flag,0,0.0,int8,2


In [10]:
print("=== FULLY NULL (100%) — 68 columns ===")
print(list(fully_null.index))

print("\n=== MOSTLY NULL (50–99%) — 3 columns ===")
print(mostly_null[["null_pct", "n_unique"]])

print("\n=== PARTIALLY NULL (0–50%) — 3 columns ===")
print(partial_null[["null_pct", "n_unique"]])

=== FULLY NULL (100%) — 68 columns ===
['PRINCIPAL_FORGIVENESS_AMOUNT', 'MISCELLANEOUS_HOLDING_EXPENSES_AND_CREDITS', 'ORIGINAL_LIST_START_DATE', 'POOL_ID', 'NON_INTEREST_BEARING_UPB', 'OTHER_FORECLOSURE_PROCEEDS', 'REPURCHASES_MAKE_WHOLE_PROCEEDS', 'CREDIT_ENHANCEMENT_PROCEEDS', 'NET_SALES_PROCEEDS', 'ASSOCIATED_TAXES_FOR_HOLDING_PROPERTY', 'ASSET_RECOVERY_COSTS', 'ARM_PRODUCT_TYPE', 'PROPERTY_PRESERVATION_AND_REPAIR_COSTS', 'FORECLOSURE_COSTS', 'DISPOSITION_DATE', 'FORECLOSURE_DATE', 'LAST_PAID_INSTALLMENT_DATE', 'UNSCHD_PRNCPL_CURR', 'TOT_SCHD_PRNCPL', 'CURR_SCHD_PRNCPL', 'ORIGINAL_LIST_PRICE', 'CURRENT_LIST_START_DATE', 'CURRENT_LIST_PRICE', 'ISSUE_SCOREB', 'MONTHS_UNTIL_FIRST_PAYMENT_RESET', 'MONTHS_BETWEEN_SUBSEQUENT_PAYMENT_RESET', 'DELINQUENT_ACCRUED_INTEREST', 'LOAN_HOLDBACK_EFFECTIVE_DATE', 'LOAN_HOLDBACK_INDICATOR', 'ZERO_BALANCE_CODE_CHANGE_DATE', 'INTEREST_RATE_CHANGE_DATE', 'FORECLOSURE_PRINCIPAL_WRITE_OFF_AMOUNT', 'PAYMENT_CHANGE_DATE', 'CUMULATIVE_CREDIT_EVENT_NET_GAIN_

In [11]:
# 1. What is zero_bal_code, and does it define default_flag?
print(df.groupby("zero_bal_code", dropna=False)["default_flag"].agg(["count", "mean"]))

# 2. MI cross-check: does MI presence match OLTV > 80?
df["_has_mi"] = df["MI_PCT"].notna()
df["_oltv_num"] = pd.to_numeric(df["OLTV"], errors="coerce")
print(pd.crosstab(df["_has_mi"], df["_oltv_num"] > 80, normalize="all").round(4))

# 3. Co-borrower cross-check
df["_has_cob"] = df["CSCORE_C"].notna()
print(pd.crosstab(df["_has_cob"], df["NUM_BO"], normalize="all").round(4))

                 count      mean
zero_bal_code                   
01             1519039  0.019017
02                1433  1.000000
03                 246  1.000000
06                1365  0.092308
09                1452  1.000000
15                 855  1.000000
16                3944  0.787272
NaN             518517  0.065151
_oltv_num   False   True 
_has_mi                  
False      0.6983  0.0014
True       0.0002  0.3002
NUM_BO         1       2       3       4    5    6
_has_cob                                          
False     0.5282  0.0009  0.0000  0.0000  0.0  0.0
True      0.0000  0.4620  0.0075  0.0013  0.0  0.0


In [12]:
#Check for hard leakage. if max_dlq_ever is present in the parquet, hard leakage
print([c for c in df.columns if "dlq" in c.lower() or "max" in c.lower()])

['max_dlq_ever']


In [17]:
# Block 1 consolidation: from 123 audited columns to a working feature set

# 1. Structural absence -> explicit indicators (do NOT impute these)
df["has_mi"] = df["MI_PCT"].notna().astype("int8") # null <-> LTV≤80, no MI required
df["has_coborrower"] = df["CSCORE_C"].notna().astype("int8") # null <-> NUM_BO==1

# 2. Column roles — the shared contract for the whole team
FEATURE_ROLES = {
    # Key: join back to portfolios. Never a feature.
    "identifier": ["LOAN_ID"],

    # Target
    "target": ["default_flag"],

    # LABEL COMPONENTS — leak by construction. EDA only, never a model input.
    "label_derived": ["max_dlq_ever", "zero_bal_code"],

    # Known at origination. Candidate model features.
    "origination": [
        "CSCORE_B", "DTI", "OLTV", "OCLTV", "NUM_BO", "ORIG_UPB", "ORIG_TERM",
        "FIRST_FLAG", "PURPOSE", "PROP", "NO_UNITS", "OCC_STAT", "STATE",
        "MSA", "ZIP", "CHANNEL", "SELLER", "MI_PCT", "MI_TYPE", "CSCORE_C",
        "has_mi", "has_coborrower", "credit_grade",
        "is_first_time", "is_homeready", "is_hfa",
        "HIGH_BALANCE_LOAN_INDICATOR", "PROPERTY_INSPECTION_WAIVER_INDICATOR",
        "RELOCATION_MORTGAGE_INDICATOR",
    ],

    # Needed for LP economics, EXCLUDED as predictors (risk-based pricing).
    "economics_only": ["ORIG_RATE", "interest_income_7yr", "lgd", "loss_if_default"],

    # Metadata / stratification, not features.
    "meta": ["orig_quarter", "ORIG_DATE", "FIRST_PAY", "ACT_PERIOD"],
}

# 3. The 68 structurally-null columns -> drop
dead_cols = fully_null.index.tolist()
df_work = df.drop(columns=dead_cols)

# 4. Sporadic missingness (~2k loans, <0.1%) -> flag, decide later
sporadic = df_work["CSCORE_B"].isna() | df_work["DTI"].isna()
print(f"Sporadic nulls (CSCORE_B or DTI): {sporadic.sum():,} loans "
      f"({sporadic.mean()*100:.3f}%) | default rate {df_work.loc[sporadic,'default_flag'].mean():.4f} "
      f"vs {df_work['default_flag'].mean():.4f} base")

# 5. The model-safe feature list, resolved against what actually exists
MODEL_FEATURES = [c for c in FEATURE_ROLES["origination"] if c in df_work.columns]
print(f"\nColumns: 128 -> {df_work.shape[1]} after dropping {len(dead_cols)} dead")
print(f"Model-safe candidate features: {len(MODEL_FEATURES)}")
print(MODEL_FEATURES)

Sporadic nulls (CSCORE_B or DTI): 1,905 loans (0.093%) | default rate 0.0399 vs 0.0341 base

Columns: 128 -> 60 after dropping 68 dead
Model-safe candidate features: 29
['CSCORE_B', 'DTI', 'OLTV', 'OCLTV', 'NUM_BO', 'ORIG_UPB', 'ORIG_TERM', 'FIRST_FLAG', 'PURPOSE', 'PROP', 'NO_UNITS', 'OCC_STAT', 'STATE', 'MSA', 'ZIP', 'CHANNEL', 'SELLER', 'MI_PCT', 'MI_TYPE', 'CSCORE_C', 'has_mi', 'has_coborrower', 'credit_grade', 'is_first_time', 'is_homeready', 'is_hfa', 'HIGH_BALANCE_LOAN_INDICATOR', 'PROPERTY_INSPECTION_WAIVER_INDICATOR', 'RELOCATION_MORTGAGE_INDICATOR']


### Check for any null columns remaining

In [19]:
df_work = df_work.drop(columns=["_has_mi", "_oltv_num", "_has_cob"], errors="ignore")
print(df_work.shape[1])
print([c for c in df_work.columns if c.startswith("_")])

57
[]


In [20]:
still_dead = [c for c in df_work.columns if df_work[c].isna().all()]
print(f"Fully-null columns remaining: {len(still_dead)} -> {still_dead}")

Fully-null columns remaining: 0 -> []


### Export cleaned data to new parquet

In [21]:
df_work = df_work.loc[~sporadic].copy() # complete-case, drops the 1,905
df_work.to_parquet("../data/processed/fannie_2017_clean.parquet")
print(f"{len(df_work):,} loans x {df_work.shape[1]} cols | "
      f"default rate {df_work['default_flag'].mean():.4f}")

2,044,946 loans x 57 cols | default rate 0.0341


# Data Quality Audit
**Purpose**: Establish that the Fannie Mae 2017 dataset is trustworthy before any modeling or optimization, and determine which of its fields are actually usable. 01_data_initial_check (from ml-lp-sim) was an initial reconnaissance for LP parameter sizing; this is the systematic audit.

**What was done:** Full-column census. All 123 columns (115 raw Fannie fields + 8 engineered in 02_initial_feat_end from ml-lp-sim) were audited for missingness, dtype, and cardinality. Columns were bucketed by null share to make a 123-row table interpretable at a glance.

**Structural inapplicability:** 68 columns are entirely null. These resolve into four coherent themes — recovery/foreclosure/disposition fields, the ARM block, time-varying current-state fields, and administrative/servicing fields — all structurally inapplicable to a performing, fixed-rate, origination-time book. This is a property of the data, not a data-quality failure. It also completes the audit trail behind the LGD decision: every field required by the Qi–Yang (2007) formula is null, so the Sirignano flat-value assumption (30% baseline / 50% downturn) is a necessary fallback, not a convenience.
**Informative absence:** Three columns appeared "mostly null" but encode structural facts rather than missing data. MI_PCT/MI_TYPE (69.97% null) are null precisely when mortgage insurance is not required — cross-checked against OLTV > 80 at 99.85% agreement. CSCORE_C (52.9% null) is null precisely when there is no co-borrower — cross-checked against NUM_BO == 1 with zero contradictions. Both were encoded as explicit indicators (has_mi, has_coborrower) rather than imputed. These cross-checks also serve as independent validation of the dataset's internal consistency.
**Label definition:** default_flag was traced to its source in src/reduce_fannie.py and documented as D180 OR credit-event disposition: (max_dlq_ever >= 6) OR (zero_bal_code ∈ {02, 03, 09, 15}). D180 is the standard convention. This closes an open decision listed in the project doc. The definition explains the observed pattern in zero_bal_code: the credit-event codes default at 100% by construction, while prepaid (1.9%) and still-active (6.5%) loans carry nonzero rates via the delinquency arm.
**Leakage identification:** Two label-derived columns survived the reduction into the feature table: max_dlq_ever and zero_bal_code. Both are components of default_flag and are unknowable at origination, when the funding decision is made. Both are fenced from modeling and retained for EDA only.
**Sporadic missingness:** Only two modeling features carry genuine missingness — CSCORE_B (0.08%) and DTI (0.02%), 1,905 loans combined. Tested for informativeness: 3.99% default rate vs. 3.41% base (1.17× lift) — directionally sensible but immaterial against the ~10× swing of CSCORE_B and on a slice this small. Dropped complete-case.

**Outputs:** A working set of 57 columns (123 -> 68 dropped), 29 model-safe candidate features, and a FEATURE_ROLES contract (identifier / target / label-derived / origination / economics-only / meta) shared with the optimization workstream. Persisted to fannie_2017_block1_clean.parquet.

**Why this serves the project goal:** The claim under test is that calibrated default probabilities produce better funding decisions than a naive rule. That claim rests on the probabilities being honest, which rests on the data being understood. This audit establishes what the data contains, proves it is internally consistent, documents what the target actually measures, and fences the fields that would have silently invalidated the model.